# 42 — Responsibility Detection
**Goal:** Extract key responsibilities and action items from JDs.

Responsibilities are the "what you'll do" half of a JD: bullet points like "Develop and deploy ML models at scale". Structurally they are **action phrases** — a verb plus an object — and that structure is exactly what this chapter extracts, using spaCy POS tagging (Ch. 10) and dependency relations (Ch. 11) to find verb → object pairs. The chapter also reads a second signal from the JD: the **seniority level** it implies, via keyword matching.

**Why it matters for resumes / ATS:** a resume bullet and a JD responsibility match when their action+object cores align — "reduced inference latency" (resume) vs "optimize model latency" (JD). Extracting the JD side of that pair is a prerequisite for Ch. 46's matching. Seniority is a coarser but crucial filter: a senior posting should not be matched against mid-level resumes, and keyword signals like "mentor" or "lead" often carry that information.

## 1. Identifying Action Phrases

A responsibility bullet is an **action phrase**: a verb ("develop", "deploy", "mentor") and usually an object ("ML models", "junior data scientists"). spaCy tags each token's part of speech and records dependency relations, so the extractor can walk the verbs and read each verb's object off its dependency children — no regex, no manual rules.

**What the code does:**
- Loads `en_core_web_sm` and parses a five-bullet responsibilities sample.
- For every token with `pos_ == "VERB"`, looks for a child with a direct-object label (`dobj`, `pobj`, or `attr`).
- Prints the verb's **lemma** (so "deploying"/"deployed" normalize to `deploy`) and the object, or `(none)`.

**Expected:** running this on the sample yields `develop -> (none)`, `deploy -> models`, `collaborate -> (none)`, and `validate -> findings`. Two honest imperfections to learn from: (1) `develop` and `collaborate` show no object because their real objects sit behind conjunctions/prepositions ("collaborate *with* teams"); (2) bullet-initial capitalized verbs like "Mentor" and "Design" get tagged as nouns/adjectives by the tagger and are skipped entirely. The parser also mis-attaches "hypotheses" as a modifier of "findings" — dependency output is useful but not oracle-accurate on terse bullet text.

In [ ]:
import re, spacy
nlp = spacy.load("en_core_web_sm")

responsibilities_text = """Responsibilities
- Develop and deploy ML models at scale
- Collaborate with cross-functional teams
- Mentor junior data scientists
- Design experiments to validate hypotheses
- Present findings to stakeholders
"""

doc = nlp(responsibilities_text)
print("Action verb detection:")
for token in doc:
    if token.pos_ == "VERB":
        # Find the object
        obj = next((child.text for child in token.children if child.dep_ in ("dobj", "pobj", "attr")), None)
        print(f"  Action: {token.lemma_:12s} -> Object: {obj or '(none)'}")

## 2. Seniority Signal Detection

Job level is usually stated in the JD — "Senior", "Lead", "5+ years" — and `detect_seniority()` picks the level with a keyword sweep: a dict of level → signal phrases, checked in order, first hit wins.

**What the code does:**
- `seniority_signals` maps four levels (junior / mid / senior / manager) to phrase lists ("entry level", "3-5 years", "principal", "head of", …).
- `detect_seniority()` lowercases the whole JD once, then scans every level's signals in dict order with a plain substring `in` test.
- Returns the first level whose signal appears, or `"not specified"` if none do.

**Expected:** on the sample JD this returns **`junior`** — not because the role is junior, but because the responsibilities line "Mentor **junior** data scientists" contains "junior", and the `junior` level is checked *before* `senior` in dict order. The title says "Senior Data Scientist", yet the keyword scan never gets there. That is the classic ordering trap of first-match-wins keyword logic: signal priority is baked into dict order, and a stray mention anywhere in the document can override the header. Ordering signals by specificity (or scanning the header first) would fix it.

In [ ]:
seniority_signals = {
    "junior": ["junior", "early career", "0-2 years", "entry level"],
    "mid": ["mid", "3-5 years", "experienced"],
    "senior": ["senior", "lead", "staff", "principal", "5+ years", "7+ years"],
    "manager": ["manager", "head of", "director", "lead a team", "manage"],
}

def detect_seniority(jd_text):
    text_lower = jd_text.lower()
    for level, signals in seniority_signals.items():
        for signal in signals:
            if signal in text_lower:
                return level
    return "not specified"

print(f"Seniority: {detect_seniority(jd)}")

## Summary: POS tagging extracts action verbs. Keyword matching detects seniority.

**Responsibilities are action+object pairs in disguise — and both extraction routes here are fast, transparent, and imperfect in instructive ways.**

POS+dependency extraction recovers `deploy -> models` cleanly but misses verbs the tagger mislabels and objects buried behind prepositions. Keyword seniority detection is a one-liner that works — until a single "junior" in a mentoring bullet beats "Senior" in the title, because dict order *is* priority order. Neither result is production-grade on its own; both are the right first pass. The extracted action pairs are exactly what Ch. 46 will align against resume achievement bullets, and the seniority flag can gate which resumes are even compared. Ch. 43 completes the JD picture by mining qualifications.